# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gulgumusdere/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN").strip()

import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

fact_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
print(con.sql(f"DESCRIBE SELECT * FROM '{fact_path}' LIMIT 1"))

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 1. My rule and its reason codes
**Signal 1 — CTR vs. position tier: CONFIRMED.** Mean CTR declines from top_3 to deep (0.0038 → 0.0004), roughly a 9x drop, with n = 197K–508K in the largest buckets. Median CTR was 0 in every bucket because at this daily grain, 62–93% of rows per tier have zero clicks — so mean_ctr was used instead of median_ctr.

**Signal 2 — CTR vs. impression volume: MIXED.** Mean CTR is nearly flat across buckets (0.0028–0.0031), only a small, noisy decline at higher volume. Not strong enough to anchor a rule on its own.

**Rule (plain words):** Flag content items whose actual CTR is far below the mean CTR of their position tier, among rows with enough impressions (≥50) and usable GSC data (gsc_data_available IS TRUE).

**Reason codes:**
- `ctr_far_below_position_peers` — gap is large (threshold set in Section 2)
- `ctr_below_position_peers` — gap is moderate

**Action label:** `snippet_review`

In [3]:
q_signal = f"""
SELECT
    CASE
        WHEN gsc_sum_position / gsc_impressions <= 3 THEN 'top_3'
        WHEN gsc_sum_position / gsc_impressions <= 10 THEN 'page_1'
        WHEN gsc_sum_position / gsc_impressions <= 20 THEN 'page_2'
        WHEN gsc_sum_position / gsc_impressions <= 50 THEN 'page_3_5'
        ELSE 'deep'
    END AS position_tier,
    COUNT(*) AS n,
    ROUND(MEDIAN(gsc_clicks * 1.0 / gsc_impressions), 6) AS median_ctr,
    ROUND(AVG(gsc_clicks * 1.0 / gsc_impressions), 6) AS mean_ctr,
    ROUND(SUM(CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END) * 1.0 / COUNT(*), 4) AS zero_click_share
FROM '{fact_path}'
WHERE gsc_data_available IS TRUE
  AND gsc_impressions >= 50
GROUP BY position_tier
ORDER BY MIN(gsc_sum_position / gsc_impressions)
"""
signal_result = con.sql(q_signal).df()
print(signal_result)

  position_tier       n  median_ctr  mean_ctr  zero_click_share
0         top_3  197354         0.0  0.003785            0.6213
1        page_1  507890         0.0  0.003347            0.6389
2        page_2  138253         0.0  0.003142            0.7072
3      page_3_5  184964         0.0  0.001566            0.7770
4          deep    8981         0.0  0.000408            0.9316


In [4]:
q_volume = f"""
SELECT
    CASE
        WHEN gsc_impressions < 100 THEN 'low_50_99'
        WHEN gsc_impressions < 500 THEN 'mid_100_499'
        WHEN gsc_impressions < 2000 THEN 'high_500_1999'
        ELSE 'very_high_2000plus'
    END AS volume_bucket,
    COUNT(*) AS n,
    ROUND(AVG(gsc_clicks * 1.0 / gsc_impressions), 6) AS mean_ctr
FROM '{fact_path}'
WHERE gsc_data_available IS TRUE
  AND gsc_impressions >= 50
GROUP BY volume_bucket
ORDER BY MIN(gsc_impressions)
"""
volume_result = con.sql(q_volume).df()
print(volume_result)

        volume_bucket       n  mean_ctr
0           low_50_99  398834  0.003070
1         mid_100_499  537157  0.003101
2       high_500_1999   93794  0.002804
3  very_high_2000plus    7657  0.002805


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
q_queue = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(gsc_sum_position) AS sum_position
FROM '{fact_path}'
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 50
"""
queue_df = con.sql(q_queue).df()

queue_df["ctr"] = queue_df["clicks"] / queue_df["impressions"]
queue_df["weighted_position"] = queue_df["sum_position"] / queue_df["impressions"]

import pandas as pd

queue_df["position_tier"] = pd.cut(
    queue_df["weighted_position"],
    bins=[0, 3, 10, 20, 50, 100000],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"]
)

tier_mean_ctr = queue_df.groupby("position_tier", observed=True)["ctr"].mean()
queue_df["expected_ctr"] = queue_df["position_tier"].map(tier_mean_ctr.to_dict()).astype(float)
queue_df["ctr_gap"] = queue_df["expected_ctr"] - queue_df["ctr"]

# Threshold: only positive gaps (underperforming) get a reason code.
# far_below = top quartile of positive gaps; below = rest.
gap_q75 = queue_df.loc[queue_df["ctr_gap"] > 0, "ctr_gap"].quantile(0.75)

def reason_code(gap):
    if gap <= 0:
        return None
    elif gap >= gap_q75:
        return "ctr_far_below_position_peers"
    else:
        return "ctr_below_position_peers"

queue_df["reason_code"] = queue_df["ctr_gap"].apply(reason_code)
queue_df["action"] = queue_df["reason_code"].apply(lambda r: "snippet_review" if r else "monitor")
queue_df["score"] = queue_df["ctr_gap"]

ranked = queue_df.sort_values("score", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Total rows in queue: {len(ranked)}")
print(f"Rows flagged (score > 0): {(ranked['score'] > 0).sum()}")
ranked.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows in queue: 116114
Rows flagged (score > 0): 81579


,client_hash_id,content_hash_id,impressions,clicks,sum_position,ctr,weighted_position,position_tier,expected_ctr,ctr_gap,reason_code,action,score
0,client_20259bd6705d81d4,content_027dd55319cee021,93.0,0.0,205.0,0.0,2.204301,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
1,client_23a62021009f63c4,content_ab7b026f01dc4fe6,980.0,0.0,529.0,0.0,0.539796,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
2,client_20259bd6705d81d4,content_a6186f261755d32c,485.0,0.0,942.0,0.0,1.942268,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
3,client_23a62021009f63c4,content_5c1e579ae6d2fc3e,359.0,0.0,879.0,0.0,2.448468,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
4,client_fef1a8f436438636,content_2725b2000a14dba7,359.0,0.0,584.0,0.0,1.626741,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
5,client_fef1a8f436438636,content_d00c26e8983b4cb0,1030.0,0.0,1415.0,0.0,1.373786,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
6,client_73cda7b4e4f265ea,content_53dde43c1721ecf2,53.0,0.0,12.0,0.0,0.226415,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
7,client_3f0ce4d44fe94f3d,content_91e853906588c189,62.0,0.0,34.0,0.0,0.548387,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
8,client_73cda7b4e4f265ea,content_231bf5a10d13e4bc,300.0,0.0,531.0,0.0,1.770000,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
9,client_73cda7b4e4f265ea,content_bb4d03df598d0869,313.0,0.0,229.0,0.0,0.731629,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463


In [7]:
q_queue = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(gsc_sum_position) AS sum_position
FROM '{fact_path}'
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 50
"""
queue_df = con.sql(q_queue).df()

queue_df["ctr"] = queue_df["clicks"] / queue_df["impressions"]
queue_df["weighted_position"] = queue_df["sum_position"] / queue_df["impressions"]

import pandas as pd

queue_df["position_tier"] = pd.cut(
    queue_df["weighted_position"],
    bins=[0, 3, 10, 20, 50, 100000],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"]
)

tier_mean_ctr = queue_df.groupby("position_tier", observed=True)["ctr"].mean()
queue_df["expected_ctr"] = queue_df["position_tier"].map(tier_mean_ctr.to_dict()).astype(float)
queue_df["ctr_gap"] = queue_df["expected_ctr"] - queue_df["ctr"]

# Threshold: only positive gaps (underperforming) get a reason code.
# far_below = top decile of positive gaps; below = the rest of the upper half.
gap_q90 = queue_df.loc[queue_df["ctr_gap"] > 0, "ctr_gap"].quantile(0.90)
gap_q50 = queue_df.loc[queue_df["ctr_gap"] > 0, "ctr_gap"].quantile(0.50)

def reason_code(gap):
    if gap <= 0:
        return None
    elif gap >= gap_q90:
        return "ctr_far_below_position_peers"
    elif gap >= gap_q50:
        return "ctr_below_position_peers"
    else:
        return None

queue_df["reason_code"] = queue_df["ctr_gap"].apply(reason_code)
queue_df["action"] = queue_df["reason_code"].apply(lambda r: "snippet_review" if r else "monitor")
queue_df["score"] = queue_df["ctr_gap"]

ranked = queue_df.sort_values("score", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Total rows in queue: {len(ranked)}")
print(f"Rows flagged (score > 0, above median gap): {(ranked['action'] == 'snippet_review').sum()}")
ranked.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows in queue: 116114
Rows flagged (score > 0, above median gap): 40790


,client_hash_id,content_hash_id,impressions,clicks,sum_position,ctr,weighted_position,position_tier,expected_ctr,ctr_gap,reason_code,action,score
0,client_e547b89c05043229,content_4f4a00e5865b998c,228.0,0.0,375.0,0.0,1.644737,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
1,client_73cda7b4e4f265ea,content_769517934497fdde,289.0,0.0,287.0,0.0,0.993080,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
2,client_e547b89c05043229,content_d3e61f814d0fceb2,62.0,0.0,100.0,0.0,1.612903,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
3,client_73cda7b4e4f265ea,content_520e203a08cd69ee,10462.0,0.0,2402.0,0.0,0.229593,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
4,client_e547b89c05043229,content_70928e26fb52a498,795.0,0.0,2235.0,0.0,2.811321,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
5,client_73cda7b4e4f265ea,content_d397987113cb84a0,9887.0,0.0,20006.0,0.0,2.023465,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
6,client_e547b89c05043229,content_b9dbe78fa73595ce,581.0,0.0,1223.0,0.0,2.104991,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
7,client_fef1a8f436438636,content_7d0ed5108074bd79,564.0,0.0,479.0,0.0,0.849291,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
8,client_3f0ce4d44fe94f3d,content_719c63e2df265aa8,756.0,0.0,1562.0,0.0,2.066138,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463
9,client_fef1a8f436438636,content_28d4c4316572631b,907.0,0.0,2064.0,0.0,2.275634,top_3,0.003463,0.003463,ctr_far_below_position_peers,snippet_review,0.003463


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.